# litterbug — Colab walkthrough

Full pipeline in order: GPU check, install, data, validate, EDA, train both models, evaluate, error
analysis, inference.

Needs a GPU runtime (`Runtime -> Change runtime type -> T4 GPU`) and about three hours. `dataset/`,
`runs/` and `*.pt` are git-ignored, so the data and the checkpoints are both produced here.

## 1. Confirm the GPU

Ultralytics falls back to CPU silently, which turns three hours into several days.

In [1]:
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("vram:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    raise SystemExit("No GPU — set Runtime -> Change runtime type -> T4 GPU and re-run.")

torch 2.14.0+cu130
cuda available: True
device: NVIDIA GeForce RTX 3060 Laptop GPU
vram: 6.1 GB


## 2. Clone and install

The install is **editable** on purpose. `PROJECT_ROOT` in `src/litterbug/common/constants.py` is
`Path(__file__).resolve().parents[3]`, so only an editable install keeps `dataset/` and `runs/`
resolving inside the clone.

An editable install is only a `.pth` file, and Python reads those when an interpreter **starts**,
not when pip writes them. This kernel started before the install, so the cell also puts `src/` on
`sys.path` by hand. Without that, `import litterbug` finds something else entirely: Colab keeps
`/content` on `sys.path` and the clone is itself named `litterbug`, so the clone directory resolves
as a PEP 420 namespace package. Its only child is `src/`, so `litterbug.common` does not exist under
it and the import fails with `No module named 'litterbug.common'`. Subprocesses are unaffected —
`!litterbug ...` starts a fresh interpreter, which does read the `.pth`.


In [2]:
import importlib
import sys
from pathlib import Path

REPO_URL = "https://github.com/strawberyy-coconut/litterbug.git"
WORKDIR = Path("/content/litterbug")

%cd /content
# Reuse a checkout only if it is a clone, and fast-forward it: one left behind by an
# earlier session predates the current packaging layout and fails the import below.
!test -d {WORKDIR}/.git || (rm -rf {WORKDIR} && git clone --depth 1 {REPO_URL} {WORKDIR})
%cd {WORKDIR}
!git fetch --depth 1 --quiet origin && git reset --hard --quiet FETCH_HEAD
%pip install -q -e .

# The editable install is a `.pth` file and Python reads those at interpreter startup, so it is
# not on this kernel's sys.path yet. Without this, `litterbug` resolves to the clone directory
# itself - a namespace package whose only child is `src/`, hence no `common`.
SRC = WORKDIR / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
importlib.invalidate_caches()


[Errno 2] No such file or directory: '/content'
/workspaces/litterbug/notebooks
Cloning into 'litterbug'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 76 (delta 1), reused 63 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 15.62 MiB | 23.56 MiB/s, done.
Resolving deltas: 100% (1/1), done.
[Errno 2] No such file or directory: '/content/litterbug'
/workspaces/litterbug/notebooks
/workspaces/litterbug/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Paths must resolve into the clone, not into site-packages.
from litterbug.common.constants import CLASS_NAMES, DATA_YAML, PROJECT_ROOT, RUNS_DIR

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_YAML    :", DATA_YAML)
print("RUNS_DIR     :", RUNS_DIR)
print("classes      :", CLASS_NAMES)

assert "site-packages" not in str(PROJECT_ROOT), "not an editable install - re-run the cell above"

In [ ]:
# Resolved configuration and environment, no GPU time.
!litterbug train --task segment --dry-run

## 3. Fetch the dataset

BUU Waste Occlusion Dataset, VisionLab, Burapha University — **CC BY 4.0** (citation:
`report/report.md` §11).

The 2,000 images are deliberately absent from this repository: `dataset/` is git-ignored, and the
archive lives on Kaggle. Kaggle requires a signed-in account to download a dataset through its API at
all, public ones included, so this one step needs a credential that belongs to you and not to the
repo.

Get it from <https://www.kaggle.com/settings/api> → **Generate New Token**. That gives a single token
string. Older guides describe downloading a `kaggle.json` instead; Kaggle now files that under
"Legacy API Credentials" (it still works — it is just no longer what the settings page hands you
first). The cell uses the first of these it finds:

1. a Colab **secret** named `KAGGLE_API_TOKEN` (key icon in the left sidebar) — recommended, since
   nothing is then typed into a cell;
2. the `KAGGLE_API_TOKEN` environment variable;
3. a legacy `~/.kaggle/kaggle.json`, so an existing setup keeps working;
4. otherwise it prompts — and Colab cannot suppress the echo on that prompt, so prefer 1.

The token is only exported for the download subprocess: the cell never writes it to the repo and
never prints it. Re-running the cell is a no-op once `data.yaml` is in place.


In [ ]:
import os
import shutil
from contextlib import suppress
from getpass import getpass
from pathlib import Path

DATA_DIR = WORKDIR / "dataset" / "1_Model_Training_Data"
ARCHIVE = Path("/content/kaggle")
LEGACY_CREDENTIALS = Path.home() / ".kaggle" / "kaggle.json"


def colab_secret(name: str) -> str | None:
    """Value of a Colab secret, or None when there is no Colab or no such secret."""
    with suppress(Exception):
        from google.colab import userdata

        return userdata.get(name)
    return None


if not (DATA_DIR / "data.yaml").is_file():
    %pip install -q kaggle

    # The token is a credential: read it, hand it to the download, store it nowhere.
    token = colab_secret("KAGGLE_API_TOKEN") or os.environ.get("KAGGLE_API_TOKEN")
    if not token and not LEGACY_CREDENTIALS.is_file():
        token = getpass("Kaggle API token (https://www.kaggle.com/settings/api): ").strip()
        if not token:
            raise SystemExit("No Kaggle credentials - cannot download the dataset.")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token

    shutil.rmtree(ARCHIVE, ignore_errors=True)
    ARCHIVE.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d visionlab1buu/buu-waste-occlusion-dataset -p {ARCHIVE} --unzip

    # The archive keeps the provider's layout; link data.yaml into the path the pipeline expects.
    found = [p.parent for p in ARCHIVE.rglob("data.yaml")]
    if not found:
        raise FileNotFoundError(f"No data.yaml under {ARCHIVE}; arrange it as {DATA_DIR}.")
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    if not DATA_DIR.exists():
        DATA_DIR.symlink_to(found[0])

print("dataset:", DATA_DIR.resolve())


## 4. Validate the data

Image/label pairing, polygon validity, coordinate ranges, class balance. Must reproduce 1,400 / 400 /
200 images and 17,400 / 4,941 / 2,507 instances.

In [ ]:
!litterbug validate

## 5. Exploratory analysis

Class counts, imbalance, geometry and lighting. Writes `runs/dataset-eda/dataset_eda.json`.

In [ ]:
!litterbug eda

## 6. Fine-tune both models

Same split, schedule and seed, so the head is the only difference. Hyperparameters are frozen in
`src/litterbug/common/config.py`; a one-epoch run is a plumbing check, not a preview.

In [ ]:
!litterbug train --task segment --name litterbug-segment-yolo26s-seg
!litterbug train --task detect --name litterbug-detect-yolo26s

## 7. Evaluate

`valid` is for iteration. `test` is touched once, at the end.

In [ ]:
from pathlib import Path


def newest_checkpoint(task: str) -> Path:
    """Most recent best.pt for a task, so the notebook does not hardcode a run name."""
    candidates = sorted(
        Path(RUNS_DIR).glob(f"*{task}*/weights/best.pt"), key=lambda p: p.stat().st_mtime
    )
    if not candidates:
        raise FileNotFoundError(f"no checkpoint under {RUNS_DIR} for {task!r}")
    return candidates[-1]


SEGMENT = newest_checkpoint("seg")
DETECT = newest_checkpoint("detect")
print("segment:", SEGMENT)
print("detect :", DETECT)

In [ ]:
!litterbug val --weights {SEGMENT} --split val

In [ ]:
# The held-out split. Run once.
!litterbug val --weights {SEGMENT} --split test
!litterbug val --weights {DETECT} --split test

## 8. Error analysis

Re-derives the per-instance matching Ultralytics does not expose: greedy, one-to-one, class-agnostic,
at mask IoU 0.5. `--iou-mode box` matches on boxes instead.

In [ ]:
!litterbug analyze --weights {SEGMENT} --split test
!litterbug analyze --weights {SEGMENT} --split test --iou-mode box

In [ ]:
from IPython.display import Image, Markdown, display

from litterbug.training.error_examples import ExampleConfig, examples

gallery = examples(ExampleConfig(weights=SEGMENT, split="test"))
display(Markdown(Path(gallery["commentary"]).read_text(encoding="utf-8")[:2000]))

first = sorted(Path(gallery["figures_dir"]).glob("*.png"))[0]
display(Image(filename=str(first)))

## 9. Inference

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from litterbug.running.predict import PredictConfig, predict

# First image, by name, from the held-out split.
sample = min(p for p in (DATA_DIR / "test" / "images").iterdir() if p.is_file())
result = predict(PredictConfig(weights=SEGMENT, source=sample, name="notebook-sample"))

print("kind       :", result.kind)
print("detections :", len(result.detections))
print("counts     :", result.counts)
print("artifacts  :")
for path in result.artifacts:
    print("  ", path)

# Show the artifacts, not just their names: the annotated frame inline, text and JSON as text.
IMAGES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PREVIEW = 2000
for path in map(Path, result.artifacts):
    if path.suffix.lower() in IMAGES:
        display(Image(filename=str(path)))


Video and tracking (`--track`, ByteTrack) are CLI-only: the report's clips are licensed previews that
cannot be redistributed. Results in `report/report.md` §8.

## 10. Reference results

**Segmentation, `test` — 200 images / 2,507 instances**

| Metric | Box | Mask |
| --- | --- | --- |
| mAP50-95 | 0.837 | 0.796 |
| mAP50 | 0.954 | 0.955 |
| Precision | 0.939 | 0.944 |
| Recall | 0.901 | 0.905 |

Mean IoU over matched pairs: 0.901 mask, 0.924 box.

**Detection baseline, `test`:** box mAP50-95 0.8376, mAP50 0.9557, precision 0.944, recall 0.913.

Box-head difference between the two models: 0.0004. `valid` runs one to two points higher (box 0.857 /
mask 0.814), nearly all of the gap in the smallest size quartile.